# BOCSAR Crime Data Cleaning

## Overview
This notebook cleans and processes suburb-level crime data from BOCSAR (Bureau of Crime Statistics and Research NSW) for use in the Sydney Liveability AI project.

## Data Source
- **File:** `SuburbData25Q4.csv`
- **Downloaded from:** [BOCSAR Open Datasets](https://bocsar.nsw.gov.au/statistics-dashboards/open-datasets.html)
- **Type:** Recorded Criminal Incidents by month — by Suburb

## What This Notebook Does
1. Loads the raw suburb-level crime CSV
2. Filters for the 5 MVP suburbs: Newtown, Glebe, Redfern, Surry Hills, Haymarket
3. Selects the most recent 24 months of data (Jan 2024 – Dec 2025)
4. Calculates `annual_avg` — the monthly average incident count per offence category
5. Exports the cleaned data to `data/processed/bocsar_clean.csv`

## Output
`data/processed/bocsar_clean.csv` with columns:
- `suburb` — suburb name
- `offence_category` — crime category
- `subcategory` — specific offence type
- `annual_avg` — average monthly incidents over 24 months

## Notes
- Suburb-level data was available from BOCSAR, so SA4-level mapping was not required.
- `annual_avg` is calculated from Jan 2024 to Dec 2025 (24 months).

In [1]:
import pandas as pd
import os

# load csv
base_path = os.path.expanduser("~/Desktop/36118")
df = pd.read_csv(os.path.join(base_path, "data/raw/bocsar/SuburbData25Q4.csv"))

print("Shape:", df.shape)

col1 = pd.DataFrame(df.columns.tolist(), columns=['Columns'])
col2 = pd.DataFrame(df['Suburb'].unique(), columns=['Unique Suburbs'])

display(pd.concat([col1, col2], axis=1))

FileNotFoundError: [Errno 2] No such file or directory: '/Users/nelkitchavezcalona/Desktop/36118/data/raw/bocsar/SuburbData25Q4.csv'

In [ ]:
# Five suburb filters
suburbs = ['Newtown', 'Glebe', 'Redfern', 'Surry Hills', 'Haymarket']
df_filtered = df[df['Suburb'].isin(suburbs)]

print("Shape after filter:", df_filtered.shape)
print("Suburbs found:", df_filtered['Suburb'].unique())

In [ ]:
# Select the most recent 24 months
month_cols = [col for col in df_filtered.columns if col not in ['Suburb', 'Offence category', 'Subcategory']]

# Convert column names to datetime for sorting
month_dates = pd.to_datetime(month_cols, format='%b %Y')

# Get the most recent 24 months
recent_24 = [month_cols[i] for i, d in enumerate(month_dates) if d >= month_dates[-24]]

pd.DataFrame(recent_24, columns=['Month'])

In [ ]:
# Calculate annual average from recent 24 months
df_recent = df_filtered[['Suburb', 'Offence category', 'Subcategory'] + recent_24].copy()

df_recent['annual_avg'] = df_recent[recent_24].mean(axis=1)

print(df_recent[['Suburb', 'Offence category', 'Subcategory', 'annual_avg']].head(10))

In [ ]:
# Keep only relevant columns
df_output = df_recent[['Suburb', 'Offence category', 'Subcategory', 'annual_avg']].copy()

# Normalise column names to snake_case
df_output.columns = ['suburb', 'offence_category', 'subcategory', 'annual_avg']

# Save to processed folder
output_path = os.path.join(base_path, "data/processed/bocsar_clean.csv")
df_output.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Shape:", df_output.shape)
print(df_output.head())